# AutoTune Studio on Google Colab

Fine-tune an LLM (LoRA / QLoRA) from a browser UI that opens in a **separate tab**. Datasets are streamed, so you never configure a row count.

> **Status: UNVERIFIED.** This notebook has **not been executed on Colab yet**. It is written
> against the implemented `Studio` API (verified locally on CPU), but installation, GPU training
> (Unsloth / QLoRA) and the Colab proxy path have not been run on a real Colab VM.
> Please report what happens.

**Before you start:** *Runtime > Change runtime type > T4 GPU* (or better).

**Security:** `studio.launch()` displays a link whose `#token=...` fragment is a session credential.
Anyone who has that link can control the studio while this kernel runs.
**Clear all outputs before you share, commit or publish this notebook**
(*Edit > Clear all outputs*), and run the shutdown cell when you are done.

## 2. Check the GPU

In [ ]:
!nvidia-smi || echo 'No NVIDIA GPU visible: switch the runtime type to a GPU (QLoRA needs CUDA).'

## 3. Install

There is no PyPI release yet. Pick one source:

* **(a)** install from the git repository (default: `https://github.com/AAB20/autotune-studio.git`). The repository includes the built browser UI. **The repository is private:** create a read-only GitHub token (fine-grained, *Contents: Read* on this repository) and store it as `GITHUB_TOKEN` in Colab *Secrets* (key icon) with notebook access. The token is never printed.
* **(b)** leave `AUTOTUNE_SOURCE` empty and upload the wheel `autotune_studio-<version>-py3-none-any.whl` (built with `uv build --wheel`, see `docs/installation.md`). The wheel includes the UI.

The preinstalled torch is kept (no reinstall). The GPU extras (`unsloth`, `bitsandbytes`, `nvidia-ml-py`) are installed separately. **Unsloth's install instructions change often:** if that step fails, follow https://github.com/unslothai/unsloth for the current Colab command. Without Unsloth, training falls back to the Hugging Face backend (QLoRA still works with `bitsandbytes`).

If pip says the session must be restarted, use *Runtime > Restart session* and continue with step 4.

In [ ]:
AUTOTUNE_SOURCE = "https://github.com/AAB20/autotune-studio.git"  # empty = install an uploaded wheel instead
INSTALL_GPU_EXTRAS = True   # unsloth, bitsandbytes, nvidia-ml-py

import subprocess, sys

def _github_token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None  # public repository or secret not configured

def pip_install(*args, secret=None):
    """Run pip; on failure show its output with the secret masked (pip may echo the URL)."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        out = (r.stdout + r.stderr)
        if secret:
            out = out.replace(secret, "***")
        print(out[-4000:])
        raise subprocess.CalledProcessError(r.returncode, "pip install")

if AUTOTUNE_SOURCE:
    token = _github_token()
    url = AUTOTUNE_SOURCE
    if token and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://x-access-token:{token}@", 1)
    pip_install(f"autotune-studio[hf,train] @ git+{url}", secret=token)
else:
    from google.colab import files
    uploaded = files.upload()  # choose autotune_studio-*.whl
    wheels = sorted(name for name in uploaded if name.endswith(".whl"))
    if not wheels:
        raise SystemExit("No .whl uploaded. Upload autotune_studio-<version>-py3-none-any.whl or set AUTOTUNE_SOURCE.")
    pip_install(f"/content/{wheels[-1]}[hf,train]")

if INSTALL_GPU_EXTRAS:
    try:
        pip_install("unsloth", "bitsandbytes", "nvidia-ml-py")
    except subprocess.CalledProcessError:
        print("GPU extras failed to install. See https://github.com/unslothai/unsloth for the current "
              "install command; the studio still runs with the Hugging Face backend.")

In [ ]:
# The browser UI ships as built files (committed in the repository and included in the wheel).
# does not contain them (they are build output), and then only the Python API works.
import importlib.resources as _res
_ui = _res.files("autotune_studio") / "web" / "index.html"
print("Browser UI bundled:", _ui.is_file())
if not _ui.is_file():
    print("The UI is missing: install the wheel (option b) or build the frontend before installing.")

## 4. Optional: Hugging Face token

Needed only for gated or private models/datasets. Add a secret named `HF_TOKEN` in the Colab sidebar (key icon) and allow notebook access. The token is never printed.

In [ ]:
import os
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok  # read by huggingface_hub / transformers
    print("HF token:", "found in Colab secrets" if _tok else "not set")
except Exception as exc:  # secret missing or access not granted
    print(f"HF token: not available ({type(exc).__name__}); only needed for gated/private repos")
finally:
    _tok = None

## 5. Create the studio and detect the runtime

Files under `/content` are lost when the runtime is recycled. To keep checkpoints and exports, set `USE_DRIVE = True`: Google Drive is mounted and used as the workspace.

In [ ]:
USE_DRIVE = False
WORKSPACE = None  # default: /content/autotune
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = "/content/drive/MyDrive/autotune"

In [ ]:
from autotune_studio import Studio

studio = Studio(workspace_dir=WORKSPACE)
env = studio.detect_environment()
hw = studio.detect_hardware()

print("runtime:", env.kind.value, "| python", env.python_version)
print("workspace:", env.workspace_dir)
print("allowed dataset roots:", env.allowed_roots)
print("internet:", {None: "not probed", True: "yes", False: "NO"}[env.internet_access])
print("CUDA:", hw.cuda_available, "| CUDA version:", hw.cuda_version, "| torch:", hw.torch_version,
      "| bf16:", hw.bf16_supported)
for g in hw.gpus:
    print(f"GPU {g.index}: {g.name}, {g.total_vram_bytes / 2**30:.1f} GiB VRAM")
print(f"RAM: {hw.ram_total_bytes / 2**30:.1f} GiB | disk free: {hw.disk_free_bytes / 2**30:.1f} GiB")
for note in env.notes + hw.notes:
    print("  note:", note)

## 6. Launch the UI

On Colab the UI is exposed through Colab's own port proxy. The proxy URL only works **in the same browser that is signed in to this Colab session**, and the kernel cannot probe it, so `publicly_verified` is `False` here by design. Click the button; the UI opens in a new tab. If the proxy is unavailable, the studio tries a cloudflared quick tunnel and then falls back to local mode. In every case the backend keeps running and the diagnostics explain what happened (see `docs/colab.md`).

In [ ]:
info = studio.launch()  # shows an "Open AutoTune Studio" button; the cell returns immediately

print("method:", info.method, "| public access verified:", info.publicly_verified)
print("backend (inside the VM):", info.local_url)
if info.public_url:
    print("public URL (without token):", info.public_url)
for d in info.diagnostics:
    print("  -", d)
# Do not print info.open_url: it contains the session token.

## 7. Optional: the same workflow from pure Python

You do not need the browser UI. This cell trains a small model on a Hugging Face dataset.
Set `RUN_PYTHON_WORKFLOW = True` to run it.

* **Zero row configuration:** you never pass a row count. The dataset is streamed. If its size is
  unknown (`rows: None`), the studio cannot turn epochs into steps, so you **must** set `max_steps`
  (otherwise `start_training()` raises `StepsRequired`). `max_steps=60` is a short smoke run.
* `method="qlora"` needs a CUDA GPU plus `unsloth` or `bitsandbytes` (installed in step 3).
* Only one training job runs at a time. You can watch it live in the UI tab while this cell waits.

In [ ]:
RUN_PYTHON_WORKFLOW = False  # set to True to train from Python

if RUN_PYTHON_WORKFLOW:
    ds = studio.load_dataset("huggingface", name="yahma/alpaca-cleaned")
    print("dataset:", ds.dataset_id, "| format:", ds.detected_format,
          "| rows:", ds.cardinality.rows, "(None = unknown, max_steps required)")

    studio.configure_model("unsloth/Llama-3.2-1B-Instruct")
    studio.configure_training(method="qlora", max_steps=60)
    steps, warnings = studio.validate_training()
    print("planned steps:", steps)
    for w in warnings:
        print("  warning:", w)

    studio.start_training()
    status = studio.wait()  # blocks until the job finishes
    print("state:", status.state.value, "| step:", status.step, "| loss:", status.loss)
    if status.error:
        print("error:", status.error)
    else:
        result = studio.export_model()  # LoRA adapter from the latest checkpoint
        print("exported", result.format, "to", result.path, f"({result.size_bytes / 2**20:.1f} MiB)")
        # Merged 16-bit model instead: studio.export_model(format="merged_16bit")

## 8. Shut down

In [ ]:
# Stops training (a checkpoint is saved first), the tunnel, the web server and the resource sampler.
studio.shutdown()
print("AutoTune Studio stopped.")